# ¿La 0.8B de verdad loopea en MATH-500? (`Qwen/Qwen3.5-0.8B`, Colab L4)

Fermion Research dice que la 0.8B se queda dando vueltas en **93.5%** de MATH-500 (llega al límite de tokens) y saca
**6.5%**; su agente la sube a **32%**. Sus pruebas de neuronas usan **greedy** (T=0), y en la 4B ya vimos que greedy
infla los loops (piloto 39% vs **0.7%** con nuestro sampling).

Aquí: los mismos **200 problemas** (estratificados por nivel), a **16k** tokens, dos veces:
* `qwen` = nuestro sampling de siempre (T 0.6, top_p 0.95, top_k 20, presence_penalty 1.5)
* `greedy` = T 0, como Fermion

Solo genera con el modelo base (nada de parches). ~30–45 min. Decide si la 0.8B + MATH-500 sirve como benchmark:
si con `qwen` hace loop duro en ≥10%, hay algo que arreglar.

Resultado final: `MyDrive/wsi_runs/FINALES/loopcheck_math500_qwen35-08b.json`. Al final el runtime se apaga solo.

In [ ]:
MODEL = "Qwen/Qwen3.5-0.8B"
N = 200          # problemas de MATH-500 (mismos para los dos samplings)
MAX_TOK = 16384
PROFILES = ["qwen", "greedy"]
RUN_NAME = "loopcheck_math500_qwen35-08b"
REPO = "/content/wsi"
BASE = "/content/base"
WORK = "/content/work"
S = f"{REPO}/scripts/discover_math500_blind.py"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q vllm sympy antlr4-python3-runtime==4.11.1 hf_transfer
!pip install -q flash-linear-attention || echo "fla opcional"
import vllm, transformers; print("vllm", vllm.__version__, "| transformers", transformers.__version__)

In [ ]:
import os, glob, shutil, json, subprocess, datetime
from google.colab import drive
drive.mount("/content/drive")
DRIVE = f"/content/drive/MyDrive/wsi_runs/{RUN_NAME}"
FINALES = "/content/drive/MyDrive/wsi_runs/FINALES"
BASE_CACHE = "/content/drive/MyDrive/wsi_runs/_base_cache"
VLLM_CACHE = "/content/drive/MyDrive/wsi_runs/_vllm_cache"
for d in (DRIVE, FINALES, BASE_CACHE, VLLM_CACHE, WORK):
    os.makedirs(d, exist_ok=True)
for prof in PROFILES:  # resume: bring back what an earlier session already generated
    src, dst = f"{DRIVE}/{prof}", f"{WORK}/{prof}"
    if os.path.isdir(src) and not os.path.isdir(dst):
        shutil.copytree(src, dst)
os.environ["WSI_BASE_CACHE_DIR"] = BASE_CACHE
os.environ["VLLM_CACHE_ROOT"] = VLLM_CACHE
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
# KV cache in bf16, not fp8: the 0.8B fits easily, and greedy should be numerically as close to Fermion's as we can get
os.environ["WSI_NO_FP8_KV"] = "1"
print("ya hecho en Drive:", sorted(os.listdir(DRIVE)) or "nada")

In [ ]:
if not os.path.exists(REPO):
    !git clone -q --depth 1 https://github.com/krexito/weight-space-index.git {REPO}
else:
    !cd {REPO} && git pull -q
COMMIT = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()
from huggingface_hub import snapshot_download
snapshot_download(MODEL, local_dir=BASE)
print("commit", COMMIT)


In [ ]:
import time, sys
# a vLLM engine left over from an interrupted cell keeps the GPU busy and the next run hangs: kill it first
!nvidia-smi --query-compute-apps=pid --format=csv,noheader | xargs -r kill -9

def run_step(cmd, env=None):
    print("\n>>>", " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env)
    buf, last = b"", 0.0
    while True:
        ch = p.stdout.read1(4096)
        if not ch:
            break
        buf += ch
        *lines, buf = buf.replace(b"\r\n", b"\n").replace(b"\r", b"\n").split(b"\n")
        for l in lines:
            s = l.decode(errors="replace").strip()
            if not s or "Attempted to load weight" in s:
                continue
            if "it/s" in s or "s/it" in s:  # tqdm: one line every 30 s
                if time.time() - last < 30:
                    continue
                last = time.time()
            print(s, flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"Step failed: {' '.join(cmd)}")

for prof in PROFILES:
    w = f"{WORK}/{prof}"
    env = {**os.environ, "WSI_SYNC_DIR": f"{DRIVE}/{prof}"}
    run_step(["python", "-u", S, "prepare", "--work", w, "--base-dir", BASE, "--model-repo", MODEL,
              "--sampling", prof, "--n-elicit", str(N), "--n-dev", "0", "--n-test", "0",
              "--max-tokens-discover", str(MAX_TOK)], env)
    # --force: here we only measure, so the loop gate must not stop the run
    run_step(["python", "-u", S, "elicit", "--work", w, "--force"], env)

In [ ]:
sys.path.insert(0, f"{REPO}/scripts")
from pathlib import Path
import math
import discover_math500_blind as m

def wilson(k, n, z=1.96):
    if n == 0:
        return [0.0, 0.0]
    p = k / n; d = 1 + z * z / n; c = p + z * z / (2 * n)
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return [round((c - h) / d, 4), round((c + h) / d, 4)]

res = {"model": MODEL, "n": N, "max_tokens": MAX_TOK, "profiles": {}}
rows = {}
for prof in PROFILES:
    w = Path(f"{WORK}/{prof}")
    s = m.score_math(w, "requests_elicit.jsonl", "outputs_elicit.jsonl")
    rows[prof] = {r["id"]: r for r in s["rows"]}
    n = s["n"]
    out = {"sampling": json.loads((w / "config.json").read_text())["sampling"], "n": n}
    for k, key in (("accuracy", "correct"), ("loop_killed", "looped_killed"), ("truncated", "truncated")):
        c = sum(r[key] for r in s["rows"])
        out[k] = round(c / n, 4)
        out[k + "_ci95"] = wilson(c, n)
    # closest to Fermion's "runs all the way to the length limit": hard loop (stopped early) or truncated
    out["loop_or_limit"] = round(sum(r["looped_killed"] or r["truncated"] for r in s["rows"]) / n, 4)
    by_lvl = {}
    for r in s["rows"]:
        b = by_lvl.setdefault(str(r["level"]), [0, 0, 0])
        b[0] += 1; b[1] += r["correct"]; b[2] += r["looped_killed"]
    out["by_level"] = {lv: {"n": b[0], "accuracy": round(b[1] / b[0], 3), "loop_killed": round(b[2] / b[0], 3)}
                       for lv, b in sorted(by_lvl.items())}
    out["median_tokens"] = sorted(r["n_tokens"] for r in s["rows"])[n // 2]
    res["profiles"][prof] = out
    print(f"{prof:7s} acc {out['accuracy']:.1%} {out['accuracy_ci95']} | loop duro {out['loop_killed']:.1%} "
          f"{out['loop_killed_ci95']} | truncado {out['truncated']:.1%} | loop o límite {out['loop_or_limit']:.1%} "
          f"| mediana {out['median_tokens']} tok")
ids = sorted(set(rows["qwen"]) & set(rows["greedy"]))
res["paired"] = {
    "n": len(ids),
    "both_loop": sum(rows["qwen"][i]["looped_killed"] and rows["greedy"][i]["looped_killed"] for i in ids),
    "greedy_only_loop": sum(rows["greedy"][i]["looped_killed"] and not rows["qwen"][i]["looped_killed"] for i in ids),
    "qwen_only_loop": sum(rows["qwen"][i]["looped_killed"] and not rows["greedy"][i]["looped_killed"] for i in ids),
}
q = res["profiles"]["qwen"]["loop_killed"]
res["verdict"] = ("usable: the 0.8B hard-loops with our sampling" if q >= m.MIN_LOOP_KILLED
                  else "not usable: with our sampling the 0.8B barely hard-loops")
print("pareado:", res["paired"])
print("veredicto:", res["verdict"])

In [ ]:
res["_final"] = {"complete": True, "model": MODEL, "commit": COMMIT, "notebook": "loopcheck_math500_08b_colab.ipynb",
                 "saved_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")}
dst = f"{FINALES}/{RUN_NAME}.json"
with open(dst + ".tmp", "w") as f:
    json.dump(res, f, indent=1)
os.replace(dst + ".tmp", dst)
print("Guardado final en Drive:", dst)

In [ ]:
# Apagar el runtime: solo si el resultado final ya está en Drive
AUTO_APAGAR = True
import time
dst = f"{FINALES}/{RUN_NAME}.json"
ok = os.path.exists(dst) and json.load(open(dst)).get("_final", {}).get("complete")
if AUTO_APAGAR and ok:
    time.sleep(90)
    from google.colab import runtime
    runtime.unassign()
else:
    print("No apago: el resultado no está en FINALES todavía.")
